# Notebook note

The saved results in `results/` are enough to remake the paper figures. I use this notebook when rerunning the CIFAR-10 MLP experiment or changing the schedule list.


# CIFAR-10 MLP dropout scheduling

This is the main CIFAR-10 MLP scheduling run. To be within the regime of applicability of our mean field theory (we take the limit of width>>length), we work with wide nets, mildly overfitting, and close enough to the signal-propagation edge for the scheduling effect to show up, while still being small enough to run across several seeds. It is important for the network to overfit as otherwise dropout might be counterproductive in the first place and so it does not make as much sense to schedule it- we'd be better off not using it at all! 



In [ ]:
from dropout_mft.paths import project_root

REPO_ROOT = project_root()

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.cuda.amp import autocast, GradScaler
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from tqdm.auto import tqdm
import contextlib
import warnings
warnings.filterwarnings('ignore')

# Performance settings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {device}")

if device == 'cuda':
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

USE_AMP = device == 'cuda'
AMP_DTYPE = torch.bfloat16 if (device == 'cuda' and torch.cuda.is_bf16_supported()) else torch.float16
USE_SCALER = USE_AMP and (AMP_DTYPE == torch.float16)

In [ ]:
# Experiment settings

N_SIMULATIONS = 25

# Dropout field settings
H_BAR = 0.1      # Mean dropout field
H_MAX = 0.2           # Max for step schedule

# Schedule choices for this run
DEPTH_MULTIPLIER = 2   # L = 3.5 × ξ (was 6)
SIGMA_B_SQ = 0.02      # Bias variance (was 0)
SIGMA_W_SQ = 1.98     # Weight variance (He = 2.0, slightly subcritical) - I explain this in the paper,
#when you have exact criticality in ReLU networks you have some analytical trouble and certain quantities are not well defined

# Edit this list to change which schedules are trained, I played with a lot others such as random sprinklings, or random amounts of dropout given an average
# but these seemed the most instructional
# Available schedules: constant, linear, reverse_linear, step,
# reverse_step, and none.
SCHEDULES = ['constant', 'step', 'reverse_step', 'linear', 'reverse_linear', 'none']

# Network architecture
WIDTH = 256
EPOCHS = 75
LEARNING_RATE = 1e-4
BATCH_SIZE = 100

# Optimizer schedule
USE_LR_SCHEDULE = True
LR_MIN = 1e-7

# Evaluation cadence
EVAL_EVERY = 1  # Evaluate test set every N epochs (1 = every epoch)

# Dataset
USE_FULL_DATASET = False
TRAIN_SIZE = None if USE_FULL_DATASET else 5000
TEST_SIZE = None if USE_FULL_DATASET else 5000
USE_GPU_DATASET = (device == 'cuda')


#Note, if you wish to try this on bigger networks, I would recommend training a small network first, and using mu P for appropriate scaling

In [ ]:
# Plot defaults - the colors I like, feel free to change

plt.style.use('seaborn-v0_8-whitegrid')

# Shared matplotlib defaults for this notebook
mpl.rcParams.update({
    # Fonts
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'DejaVu Serif', 'Computer Modern Roman'],
    'font.size': 10,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'legend.title_fontsize': 10,

    # Lines
    'lines.linewidth': 1.5,
    'lines.markersize': 4,

    # Axes
    'axes.linewidth': 0.8,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linewidth': 0.5,

    # Figure size and resolution defaults
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.05,

    # Math text defaults
    'mathtext.fontset': 'cm',

    # Legend defaults
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'legend.fancybox': False,
})

from dropout_mft.schedules import DISPLAY_NAMES
from dropout_mft.style import SCHEDULE_COLORS, schedule_color

COLORS = SCHEDULE_COLORS

def get_color(sched):
    return schedule_color(sched)

def get_display_name(sched):
    return DISPLAY_NAMES.get(sched, sched)

In [ ]:
# defining the different dropout schedules, as you can see most of them could be written in one line,
# # but I wrote them out for clarity. The step schedules are a bit more complicated, 
# #as they require the h_max parameter to determine how many layers to apply dropout to.
# #The linear schedules are normalized to have mean h_bar, so that the average dropout rate 
# #is consistent across schedules.

def get_dropout_schedule(schedule_type, depth, h_bar, h_max=None, seed=None, epoch=0):
    """Returns dropout rates for each layer.

    Available schedules:
    - 'none': No dropout (all zeros)
    - 'constant': Uniform dropout at h_bar
    - 'linear': Increasing dropout (0 → 2*h_bar)
    - 'reverse_linear': Decreasing dropout (2*h_bar → 0)
    - 'step': Dropout only in second half (h_adj in last n_drop layers)
    - 'reverse_step': Dropout only in first half (h_adj in first n_drop layers)
    """
    if schedule_type == 'none':
        return [0.0] * depth

    elif schedule_type == 'constant':
        return [h_bar] * depth

    elif schedule_type == 'linear':
        # Linearly increasing field, normalized to mean h_bar
        if depth == 1:
            return [h_bar]
        return [2*h_bar * ell / (depth - 1) for ell in range(depth)]

    elif schedule_type == 'reverse_linear':
        # Linearly decreasing field, normalized to mean h_bar
        if depth == 1:
            return [h_bar]
        return [2*h_bar * (depth - 1 - ell) / (depth - 1) for ell in range(depth)]

    elif schedule_type == 'step':
        # Late step: dropout only in the second half
        if h_max is None or h_max < h_bar:
            raise ValueError("step requires h_max >= h_bar")
        f = h_bar / h_max
        n_drop = max(1, int(np.ceil(f * depth)))
        h_adj = h_bar * depth / n_drop
        return [0.0] * (depth - n_drop) + [h_adj] * n_drop

    elif schedule_type == 'reverse_step':
        # Early step: dropout only in the first half
        if h_max is None or h_max < h_bar:
            raise ValueError("reverse_step requires h_max >= h_bar")
        f = h_bar / h_max
        n_drop = max(1, int(np.ceil(f * depth)))
        h_adj = h_bar * depth / n_drop
        return [h_adj] * n_drop + [0.0] * (depth - n_drop)

    else:
        raise ValueError(f"Unknown schedule: {schedule_type}")

#we use the paper scalings here
def compute_effective_xi(h_layers, schedule_type=None, h_bar_target=None):
    """Compute effective correlation length (kinked/ReLU class)."""
    kappa = 2 * np.sqrt(2) / (3 * np.pi)
    alpha = (3/2) * kappa ** (2/3)
    L = len(h_layers)

    damage = sum(h ** (1/3) for h in h_layers if h > 0) / L
    h_bar_actual = np.mean(h_layers)
    h_max_actual = max(h_layers) if h_layers else 0
    n_dropout = sum(1 for h in h_layers if h > 0)

    xi_inv = alpha * damage
    xi_eff = 1 / xi_inv if xi_inv > 0 else float('inf')

    return {
        'xi_eff': xi_eff,
        'h_bar': h_bar_actual,
        'h_max': h_max_actual,
        'n_dropout': n_dropout
    }

In [ ]:
# Pick the depth from the effective correlation length - the Deep Information Propagation 
# paper shows networks suffer when you go much deeper than the networks correlation length, 
# so we want to keep them at an o(1) multiple of it. 
# Correlation length for constant dropout in the kinked class
kappa = 2 * np.sqrt(2) / (3 * np.pi)
alpha = (3/2) * kappa ** (2/3)
xi_constant = 1 / (alpha * H_BAR ** (1/3))

# Depth as a multiple of the correlation length
DEPTH = int(DEPTH_MULTIPLIER * xi_constant)

print("=" * 60)
print("SETTINGS")
print("=" * 60)
print(f"σ_w² = {SIGMA_W_SQ:.4f}  (He init = 2.0)")
print(f"σ_b² = {SIGMA_B_SQ:.4f}")
print(f"h̄ = {H_BAR:.4f}  (dropout field)")
print(f"")
print(f"Correlation length (constant): ξ = {xi_constant:.2f}")
print(f"Network depth: L = {DEPTH}  ({DEPTH_MULTIPLIER}× ξ)")
print(f"Ratio L/ξ = {DEPTH/xi_constant:.2f}")
print(f"")
print(f"Regime check:")
print(f"  L/N = {DEPTH/WIDTH:.4f}")
print(f"  h̄/(L/N) = {H_BAR/(DEPTH/WIDTH):.1f}")
status = "✓ dropout dominates" if H_BAR > DEPTH/WIDTH else "!!!! non-Gaussianities may matter"
print(f"  {status}")
print("=" * 60)

In [ ]:
# Schedule diagnostics

theory = {}
print(f"{'Schedule':<15} {'h̄':>8} {'h_max':>8} {'#drop':>6} {'ξ_eff':>8} {'L/ξ':>6}")
print("-" * 60)

for sched in SCHEDULES:
    h_layers = get_dropout_schedule(sched, DEPTH, H_BAR, H_MAX, seed=0)
    stats = compute_effective_xi(h_layers, schedule_type=sched, h_bar_target=H_BAR)
    theory[sched] = {'h_layers': h_layers, **stats}

    xi_str = f"{stats['xi_eff']:.2f}" if stats['xi_eff'] < 1e6 else "∞"
    ratio = DEPTH/stats['xi_eff'] if stats['xi_eff'] < 1e6 else 0
    n_drop_str = f"{stats['n_dropout']:.1f}" if isinstance(stats['n_dropout'], float) else f"{stats['n_dropout']}"
    print(f"{sched:<15} {stats['h_bar']:>8.4f} {stats['h_max']:>8.4f} "
          f"{n_drop_str:>6} {xi_str:>8} {ratio:>6.2f}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(7, 3), sharey=True)
layers = np.arange(1, DEPTH + 1)

# Left panel: linear schedules
for sched in [s for s in SCHEDULES if s in ['none', 'constant', 'linear', 'reverse_linear']]:
    h = np.array(theory[sched]['h_layers'])
    xi = theory[sched]['xi_eff']
    xi_str = f"{xi:.1f}" if xi < 1e6 else r"$\infty$"
    ax1.plot(layers, h, '-o', color=get_color(sched), lw=1.5, ms=2.5,
             markeredgecolor='white', markeredgewidth=0.3,
             label=f"{get_display_name(sched)} ($\\xi$={xi_str})")

ax1.axhline(H_BAR, color='0.5', ls=':', alpha=0.8, lw=1)
ax1.set_xlabel('Layer $\\ell$')
ax1.set_ylabel('Dropout probability $h_\\ell$')
ax1.set_title('(a) Uniform & linear', loc='left')
ax1.legend(fontsize=7)

# Right panel: step schedules
for sched in [s for s in SCHEDULES if s in ['constant', 'step', 'reverse_step']]:
    h = np.array(theory[sched]['h_layers'])
    xi = theory[sched]['xi_eff']
    xi_str = f"{xi:.1f}" if xi < 1e6 else r"$\infty$"
    if 'step' in sched:
        ax2.step(layers, h, where='mid', color=get_color(sched), lw=2,
                 label=f"{get_display_name(sched)} ($\\xi$={xi_str})")
    else:
        ax2.plot(layers, h, '-', color=get_color(sched), lw=1.5, alpha=0.6,
                 label=f"{get_display_name(sched)} ($\\xi$={xi_str})")

ax2.axhline(H_BAR, color='0.5', ls=':', alpha=0.8, lw=1)
ax2.set_xlabel('Layer $\\ell$')
ax2.set_title('(b) Step schedules', loc='left')
ax2.legend(fontsize=7)

for ax in [ax1, ax2]:
    ax.set_xlim(0, DEPTH + 1)
    ax.set_ylim(-0.008, max(H_MAX, 2*H_BAR) * 1.15)

plt.tight_layout()
plt.savefig('dropout_schedules_2panel.pdf', format='pdf')
plt.show()

In [ ]:
# Model definition with small bias variance

class CriticalReLUNet(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, h_layers,
                 sigma_w_sq=2.0, sigma_b_sq=0.0,
                 schedule_type=None, h_bar=None, seed=None):
        super().__init__()
        self.depth = len(h_layers)
        self.sigma_w_sq = sigma_w_sq
        self.sigma_b_sq = sigma_b_sq
        self.schedule_type = schedule_type
        self.h_bar = h_bar
        self.seed = seed

        layers = [nn.Linear(input_dim, hidden_dim)]
        for _ in range(self.depth - 1):
            layers.append(nn.Linear(hidden_dim, hidden_dim))
        layers.append(nn.Linear(hidden_dim, output_dim))

        self.layers = nn.ModuleList(layers)
        self.dropouts = nn.ModuleList([nn.Dropout(p=h) for h in h_layers])
        self._init_critical()

    def _init_critical(self):
        for layer in self.layers:
            fan_in = layer.weight.shape[1]
            std_w = np.sqrt(self.sigma_w_sq / fan_in)
            nn.init.normal_(layer.weight, mean=0.0, std=std_w)

            if self.sigma_b_sq > 0:
                nn.init.normal_(layer.bias, mean=0.0, std=np.sqrt(self.sigma_b_sq))
            else:
                nn.init.zeros_(layer.bias)

    def forward(self, x):
        x = x.view(x.size(0), -1)
        for i, layer in enumerate(self.layers[:-1]):
            x = torch.relu(layer(x))
            x = self.dropouts[i](x)
        return self.layers[-1](x)

In [ ]:
# Training loop - nothing new here, just standard PyTorch training with optional AMP and gradient scaling for mixed precision. 
# The only real change in the loop is that we grab the dropout schedule from the model and use it to set the dropout probabilities for each layer.

def _autocast():
    if not USE_AMP:
        return contextlib.nullcontext()
    return autocast(enabled=True, dtype=AMP_DTYPE)

def iterate_batches(x, y, bs, shuffle=True):
    n = x.shape[0]
    idx = torch.randperm(n, device=x.device) if shuffle else torch.arange(n, device=x.device)
    for i in range(0, n, bs):
        yield x[idx[i:i+bs]], y[idx[i:i+bs]]

def get_cifar10(train_size=None, test_size=None, seed=0):
    """Load CIFAR-10 to GPU. If train_size/test_size is None, use full dataset."""
    mean = torch.tensor([0.4914, 0.4822, 0.4465]).view(1,3,1,1)
    std = torch.tensor([0.2470, 0.2435, 0.2616]).view(1,3,1,1)

    train = datasets.CIFAR10('./data', train=True, download=True)
    test = datasets.CIFAR10('./data', train=False, download=True)

    rng = np.random.RandomState(seed)

    # Use the full dataset or a controlled subsample
    if train_size is None or train_size >= len(train.data):
        tr_idx = np.arange(len(train.data))
        print(f'Using full training set: {len(tr_idx)} samples')
    else:
        tr_idx = rng.choice(len(train.data), train_size, replace=False)
        print(f'Using {len(tr_idx)} training samples')

    if test_size is None or test_size >= len(test.data):
        te_idx = np.arange(len(test.data))
        print(f'Using full test set: {len(te_idx)} samples')
    else:
        te_idx = rng.choice(len(test.data), test_size, replace=False)
        print(f'Using {len(te_idx)} test samples')

    x_tr = torch.from_numpy(train.data[tr_idx]).permute(0,3,1,2).float().div_(255)
    y_tr = torch.tensor(np.array(train.targets)[tr_idx])
    x_te = torch.from_numpy(test.data[te_idx]).permute(0,3,1,2).float().div_(255)
    y_te = torch.tensor(np.array(test.targets)[te_idx])

    if device == 'cuda':
        x_tr, y_tr = x_tr.cuda(), y_tr.cuda()
        x_te, y_te = x_te.cuda(), y_te.cuda()
        mean, std = mean.cuda(), std.cuda()

    x_tr = (x_tr - mean) / std
    x_te = (x_te - mean) / std

    return (x_tr, y_tr), (x_te, y_te)

def train_epoch(model, data, opt, crit, bs):
    model.train()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.0

    for xb, yb in iterate_batches(x, y, bs):
        opt.zero_grad(set_to_none=True)
        with _autocast():
            out = model(xb)
            loss = crit(out, yb)
        loss.backward()
        opt.step()

        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)

    return loss_sum/total, 100*correct/total

@torch.no_grad()
def evaluate(model, data, crit, bs):
    model.eval()
    x, y = data
    total, correct, loss_sum = 0, 0, 0.0

    for xb, yb in iterate_batches(x, y, bs, shuffle=False):
        with _autocast():
            out = model(xb)
            loss = crit(out, yb)
        total += xb.size(0)
        correct += (out.argmax(1) == yb).sum().item()
        loss_sum += loss.item() * xb.size(0)

    return loss_sum/total, 100*correct/total

In [ ]:
# Data loaders

print("Loading CIFAR-10...")
train_data, test_data = get_cifar10(TRAIN_SIZE, TEST_SIZE)
print(f"Train: {train_data[0].shape}, Test: {test_data[0].shape}")

In [ ]:
# Run the experiment

def run_experiments(schedules, n_sims, epochs):
    results = {}

    for sched in schedules:
        print(f"\n{'='*50}")
        print(f"Schedule: {sched}")
        print(f"{'='*50}")

        h_layers = theory[sched]['h_layers']
        histories = []

        for sim in range(n_sims):
            seed = 42 + sim
            torch.manual_seed(seed)
            np.random.seed(seed)

            model = CriticalReLUNet(
                input_dim=3072, hidden_dim=WIDTH, output_dim=10,
                h_layers=h_layers, sigma_w_sq=SIGMA_W_SQ, sigma_b_sq=SIGMA_B_SQ,
                schedule_type=sched, h_bar=H_BAR, seed=seed
            ).to(device)

            opt = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-7)
            scheduler = CosineAnnealingLR(opt, T_max=epochs, eta_min=LR_MIN) if USE_LR_SCHEDULE else None
            crit = nn.CrossEntropyLoss()

            hist = {'train_acc': [], 'test_acc': [], 'train_loss': [], 'test_loss': []}
            best_test_acc = 0.0
            best_test_loss = float('inf')

            pbar = tqdm(range(epochs), desc=f'{sched} sim {sim+1}/{n_sims}', leave=True)

            for ep in pbar:
                tr_loss, tr_acc = train_epoch(model, train_data, opt, crit, BATCH_SIZE)
                if scheduler: scheduler.step()

                if ep % EVAL_EVERY == 0 or ep == epochs - 1:
                    te_loss, te_acc = evaluate(model, test_data, crit, BATCH_SIZE)
                else:
                    te_loss, te_acc = hist['test_loss'][-1] if hist['test_loss'] else 0, \
                                      hist['test_acc'][-1] if hist['test_acc'] else 0

                # Track the best test point for each run
                if te_acc > best_test_acc:
                    best_test_acc = te_acc
                if te_loss < best_test_loss:
                    best_test_loss = te_loss

                hist['train_acc'].append(tr_acc)
                hist['test_acc'].append(te_acc)
                hist['train_loss'].append(tr_loss)
                hist['test_loss'].append(te_loss)

                # Progress bar update
                pbar.set_postfix({
                    'ep': f'{ep+1}/{epochs}',
                    'tr_L': f'{tr_loss:.3f}',
                    'tr_A': f'{tr_acc:.1f}%',
                    'te_L': f'{te_loss:.3f}',
                    'te_A': f'{te_acc:.1f}%',
                    'best': f'{best_test_acc:.1f}%'
                })

            histories.append(hist)

        # Aggregate curves across seeds
        results[sched] = {
            'train_acc': np.array([h['train_acc'] for h in histories]),
            'test_acc': np.array([h['test_acc'] for h in histories]),
            'train_loss': np.array([h['train_loss'] for h in histories]),
            'test_loss': np.array([h['test_loss'] for h in histories]),
        }

        best = results[sched]['test_acc'].max(axis=1)
        print(f"{sched}: Best test = {best.mean():.2f}% ± {best.std():.2f}%")

    return results

results = run_experiments(SCHEDULES, N_SIMULATIONS, EPOCHS)

In [ ]:
# Save the run
from dropout_mft.results import load_npz_result, save_npz_result

# Save the arrays and config needed by the plotting cells
save_data = {
    'results': results,
    'theory': theory,
    'config': {
        'SCHEDULES': SCHEDULES,
        'N_SIMULATIONS': N_SIMULATIONS,
        'EPOCHS': EPOCHS,
        'DEPTH': DEPTH,
        'WIDTH': WIDTH,
        'H_BAR': H_BAR,
        'H_MAX': H_MAX,
        'SIGMA_W_SQ': SIGMA_W_SQ,
        'SIGMA_B_SQ': SIGMA_B_SQ,
        'LEARNING_RATE': LEARNING_RATE,
        'LR_MIN': LR_MIN,
    }
}

save_npz_result(REPO_ROOT / "results" / "mlp" / "dropout_experiment_results.npz", save_data)

print("Results saved to results/mlp/dropout_experiment_results.npz")

In [ ]:
# Load a saved run
from dropout_mft.results import load_npz_result, save_npz_result
import numpy as np
import matplotlib.pyplot as plt

save_data = load_npz_result(REPO_ROOT / "results" / "mlp" / "dropout_experiment_results.npz")

results = save_data['results']
theory = save_data['theory']

# Restore config variables
SCHEDULES = save_data['config']['SCHEDULES']
N_SIMULATIONS = save_data['config']['N_SIMULATIONS']
EPOCHS = save_data['config']['EPOCHS']
DEPTH = save_data['config']['DEPTH']
WIDTH = save_data['config']['WIDTH']
H_BAR = save_data['config']['H_BAR']
H_MAX = save_data['config']['H_MAX']
SIGMA_W_SQ = save_data['config']['SIGMA_W_SQ']
SIGMA_B_SQ = save_data['config']['SIGMA_B_SQ']

print(f"Loaded results for schedules: {SCHEDULES}")
print(f"Config: L={DEPTH}, N={WIDTH}, {N_SIMULATIONS} sims, {EPOCHS} epochs")

# Plotting cells below can now use the restored state.

In [ ]:
# Four-panel training curves

fig, axes = plt.subplots(2, 2, figsize=(7, 5.5))

# Keep handles for the shared legend below the panels
lines = []
labels = []

for sched in SCHEDULES:
    c = get_color(sched)
    xi = theory[sched]['xi_eff']
    xi_str = f'{xi:.1f}' if xi < 1e6 else r'$\infty$'
    label = f'{get_display_name(sched)} ($\\xi$={xi_str})'

    # Training loss
    m = results[sched]['train_loss'].mean(0)
    s = results[sched]['train_loss'].std(0)
    epochs_plot = np.arange(5, 5 + len(m))
    line, = axes[0,0].semilogy(epochs_plot, m, color=c, lw=1.5)
    axes[0,0].fill_between(epochs_plot, np.maximum(m-s, 1e-6), m+s, color=c, alpha=0.15)

    # Store the legend handles once
    lines.append(line)
    labels.append(label)

    # Test loss
    m = results[sched]['test_loss'].mean(0)
    s = results[sched]['test_loss'].std(0)
    axes[0,1].semilogy(epochs_plot, m, color=c, lw=1.5)
    axes[0,1].fill_between(epochs_plot, np.maximum(m-s, 1e-6), m+s, color=c, alpha=0.15)

    # Training accuracy
    m = results[sched]['train_acc'].mean(0)
    s = results[sched]['train_acc'].std(0)
    epochs_full = np.arange(len(m))
    axes[1,0].plot(epochs_full, m, color=c, lw=1.5)
    axes[1,0].fill_between(epochs_full, m-s, m+s, color=c, alpha=0.15)

    # Test accuracy
    m = results[sched]['test_acc'].mean(0)
    s = results[sched]['test_acc'].std(0)
    axes[1,1].plot(epochs_full, m, color=c, lw=1.5)
    axes[1,1].fill_between(epochs_full, m-s, m+s, color=c, alpha=0.15)

# Configure the shared axes
axes[0,0].set_xlabel('Epoch')
axes[0,0].set_ylabel('Training loss')
axes[0,0].set_title('(a) Training loss', fontweight='normal', loc='left')

axes[0,1].set_xlabel('Epoch')
axes[0,1].set_ylabel('Test loss')
axes[0,1].set_title('(b) Test loss', fontweight='normal', loc='left')

axes[1,0].set_xlabel('Epoch')
axes[1,0].set_ylabel('Training accuracy (%)')
axes[1,0].set_title('(c) Training accuracy', fontweight='normal', loc='left')

axes[1,1].set_xlabel('Epoch')
axes[1,1].set_ylabel('Test accuracy (%)')
axes[1,1].set_title('(d) Test accuracy', fontweight='normal', loc='left')

# Keep one shared legend under the panels
fig.legend(lines, labels, loc='lower center', ncol=3, fontsize=8,
           framealpha=0.95, bbox_to_anchor=(0.5, -0.02))

plt.tight_layout(rect=[0, 0.08, 1, 1])  # Leave room at bottom for legend
plt.savefig('dropout_comparison_4panel.pdf', format='pdf', bbox_inches='tight')
plt.savefig('dropout_comparison_4panel.png', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Individual seed traces

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

for sched in SCHEDULES:
    c = get_color(sched)

    # Plot each seed trace
    for i in range(results[sched]['test_acc'].shape[0]):
        # Test accuracy
        axes[0].plot(results[sched]['test_acc'][i], color=c, alpha=0.5,
                     label=get_display_name(sched) if i == 0 else '', lw=0.8)
        # Test loss
        axes[1].semilogy(results[sched]['test_loss'][i], color=c, alpha=0.5,
                         label=get_display_name(sched) if i == 0 else '', lw=0.8)

axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Test accuracy (%)')
axes[0].set_title('(a) Test accuracy — individual runs', fontweight='normal', loc='left')
axes[0].legend(loc='lower right', fontsize=7, framealpha=0.95)

axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Test loss')
axes[1].set_title('(b) Test loss — individual runs', fontweight='normal', loc='left')

plt.tight_layout()
plt.savefig('individual_runs.pdf', format='pdf')
plt.savefig('individual_runs.png', dpi=300)
plt.show()

In [ ]:
# Final accuracy distribution

fig, axes = plt.subplots(1, 2, figsize=(7, 2.8))

# Collect final test accuracies
final_accs = {}
for sched in SCHEDULES:
    final_accs[sched] = results[sched]['test_acc'][:, -1]  # Last epoch

# Histogram view
ax = axes[0]
all_vals = np.concatenate(list(final_accs.values()))
bins = np.linspace(all_vals.min() - 1, all_vals.max() + 1, 20)

for sched in SCHEDULES:
    c = get_color(sched)
    vals = final_accs[sched]
    mu = vals.mean()
    ax.hist(vals, bins=bins, alpha=0.4, color=c,
            label=f'{get_display_name(sched)} ($\\mu$={mu:.1f}%)', edgecolor='white', linewidth=0.5)
    ax.axvline(mu, color=c, linestyle='--', linewidth=1.2)

ax.set_xlabel('Final test accuracy (%)')
ax.set_ylabel('Count')
ax.set_title('(a) Distribution of final accuracy', fontweight='normal', loc='left')
ax.legend(fontsize=6, framealpha=0.95, loc='upper left')

# Boxplot view
ax = axes[1]
box_data = [final_accs[sched] for sched in SCHEDULES]
box_labels = [get_display_name(sched) for sched in SCHEDULES]
bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.6)

for patch, sched in zip(bp['boxes'], SCHEDULES):
    patch.set_facecolor(get_color(sched))
    patch.set_alpha(0.6)
    patch.set_edgecolor('0.3')
    patch.set_linewidth(0.8)

for element in ['whiskers', 'caps', 'medians']:
    for item in bp[element]:
        item.set_color('0.3')
        item.set_linewidth(0.8)

ax.set_ylabel('Final test accuracy (%)')
ax.set_title('(b) Accuracy by schedule', fontweight='normal', loc='left')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('final_accuracy_distribution.pdf', format='pdf')
plt.savefig('final_accuracy_distribution.png', dpi=300)
plt.show()

In [ ]:
# Summarize the saved results

print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"{'Schedule':<20} {'ξ_eff':>8} {'L/ξ':>8} {'Best Test':>18}")
print("-"*60)

for sched in SCHEDULES:
    xi = theory[sched]['xi_eff']
    best = results[sched]['test_acc'].max(axis=1)
    n = len(best)
    xi_str = f"{xi:.2f}" if xi < 1e6 else "∞"
    ratio = f"{DEPTH/xi:.2f}" if xi < 1e6 else "0"
    print(f"{get_display_name(sched):<20} {xi_str:>8} {ratio:>8} {best.mean():>8.2f}% ± {best.std()/np.sqrt(n):.2f}%")

print("="*70)
print(f"\nConfiguration: L={DEPTH}, σ_w²={SIGMA_W_SQ}, σ_b²={SIGMA_B_SQ}")
print(f"Regime: L/N={DEPTH/WIDTH:.4f}, h̄={H_BAR}")
print(f"Schedules tested: {len(SCHEDULES)}")
print("Note: Errors are SEM")

In [ ]:
# Combined paper figure

fig, axes = plt.subplot_mosaic(
    [
        ['train_loss', 'test_loss'],
        ['train_acc', 'test_acc'],
        ['final_dist', 'final_dist'],
    ],
    figsize=(7, 7),
    gridspec_kw=dict(height_ratios=[1, 1, 0.9], hspace=0.35, wspace=0.3),
)
ax_loss_train = axes['train_loss']
ax_loss_test = axes['test_loss']
ax_acc_train = axes['train_acc']
ax_acc_test = axes['test_acc']
ax_dist = axes['final_dist']

def plot_mean_band(ax, x, mean, std, color, *, log=False, label=None):
    plot = ax.semilogy if log else ax.plot
    plot(x, mean, color=color, label=label, lw=1.5)
    lower = np.maximum(mean - std, 1e-6) if log else mean - std
    ax.fill_between(x, lower, mean + std, color=color, alpha=0.15)

def label_curve_panel(ax, title, ylabel):
    ax.set_xlabel('Epoch')
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='normal', loc='left')

# Plot the training curves
for sched in SCHEDULES:
    c = get_color(sched)
    xi = theory[sched]['xi_eff']
    xi_str = f'{xi:.1f}' if xi < 1e6 else r'$\infty$'
    label = f'{get_display_name(sched)} ($\\xi$={xi_str})'

    # Training loss
    m = results[sched]['train_loss'].mean(0)[5:]
    s = results[sched]['train_loss'].std(0)[5:]
    epochs_plot = np.arange(5, 5 + len(m))
    plot_mean_band(ax_loss_train, epochs_plot, m, s, c, log=True)

    # Test loss
    m = results[sched]['test_loss'].mean(0)[5:]
    s = results[sched]['test_loss'].std(0)[5:]
    plot_mean_band(ax_loss_test, epochs_plot, m, s, c, log=True)

    # Training accuracy
    m = results[sched]['train_acc'].mean(0)
    s = results[sched]['train_acc'].std(0)
    epochs_full = np.arange(len(m))
    plot_mean_band(ax_acc_train, epochs_full, m, s, c)

    # Test accuracy
    m = results[sched]['test_acc'].mean(0)
    s = results[sched]['test_acc'].std(0)
    plot_mean_band(ax_acc_test, epochs_full, m, s, c, label=label)

# Configure the training-curve axes
label_curve_panel(ax_loss_train, '(a) Training loss', 'Training loss')
label_curve_panel(ax_loss_test, '(b) Test loss', 'Test loss')
label_curve_panel(ax_acc_train, '(c) Training accuracy', 'Training accuracy (%)')
label_curve_panel(ax_acc_test, '(d) Test accuracy', 'Test accuracy (%)')
ax_acc_test.legend(loc='lower right', fontsize=7, framealpha=0.95)

# Bottom panel: final-accuracy boxplot
box_data = [final_accs[sched] for sched in SCHEDULES]
box_labels = [get_display_name(sched) for sched in SCHEDULES]
bp = ax_dist.boxplot(box_data, labels=box_labels, patch_artist=True, widths=0.5, vert=True)

for patch, sched in zip(bp['boxes'], SCHEDULES):
    patch.set_facecolor(get_color(sched))
    patch.set_alpha(0.6)
    patch.set_edgecolor('0.3')
    patch.set_linewidth(0.8)

for element in ['whiskers', 'caps', 'medians']:
    for item in bp[element]:
        item.set_color('0.3')
        item.set_linewidth(0.8)

ax_dist.set_ylabel('Final test accuracy (%)')
ax_dist.set_title('(e) Final accuracy distribution', fontweight='normal', loc='left')
ax_dist.tick_params(axis='x', rotation=20)

plt.savefig('comprehensive_figure.pdf', format='pdf')
plt.savefig('comprehensive_figure.png', dpi=300)
plt.show()

In [ ]:
# Correlation length versus accuracy

fig, ax = plt.subplots(figsize=(5.5, 4.5))

for sched in SCHEDULES:
    xi_eff = theory[sched]['xi_eff']
    final_acc = results[sched]['test_acc'][:, -1]

    if xi_eff == float('inf') or xi_eff > 1e6:
        continue

    ax.errorbar(xi_eff, final_acc.mean(), yerr=final_acc.std(),
                fmt='o', markersize=11, capsize=4, capthick=1.5,
                color=get_color(sched), markeredgecolor='white', markeredgewidth=1.2,
                label=get_display_name(sched), zorder=5)

# No-dropout baseline reference
none_acc = results['none']['test_acc'][:, -1].mean()
none_std = results['none']['test_acc'][:, -1].std()
ax.axhline(none_acc, color=get_color('none'), linestyle='-', alpha=0.8, lw=1.5, zorder=1)
ax.axhspan(none_acc - none_std, none_acc + none_std, color=get_color('none'), alpha=0.15, zorder=0)
ax.text(ax.get_xlim()[0] + 0.1, none_acc + 1.5,
        f'No dropout ($\\xi \\to \\infty$): {none_acc:.1f}%',
        fontsize=9, color=get_color('none'), va='bottom')

# Depth reference line
ax.axvline(DEPTH, color='0.5', ls=':', lw=1, alpha=0.7, zorder=0)
ax.text(DEPTH + 0.2, ax.get_ylim()[0] + 1, f'$L={DEPTH}$', fontsize=8, color='0.5', rotation=90, va='bottom')

ax.set_xlabel(r'Effective correlation length $\xi_{\rm eff}$')
ax.set_ylabel('Final test accuracy (%)')
ax.set_title(r'Correlation length vs empirical performance')

ax.legend(loc='lower right', fontsize=9, framealpha=0.95, edgecolor='0.8')
ax.set_xlim(left=2.5)

plt.tight_layout()
plt.savefig('xi_eff_vs_accuracy.png', dpi=300, bbox_inches='tight')
plt.savefig('xi_eff_vs_accuracy.pdf', format='pdf', bbox_inches='tight')
plt.show()